# 0B — Train / Validation Split & Evaluation

Defines the temporal split and the MAP@12 metric you'll use for every model this week.  
Also saves the split datasets so other notebooks can just load them.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/MLII_Final/data/parquet")
OUT_DIR  = Path("/content/drive/MyDrive/MLII_Final/data/split")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load transactions

In [7]:
transactions = pd.read_parquet(DATA_DIR / "transactions.parquet")
print(f"Total transactions: {len(transactions):,}")
print(f"Date range: {transactions['t_dat'].min()} → {transactions['t_dat'].max()}")

Total transactions: 31,788,324
Date range: 2018-09-20 00:00:00 → 2020-09-22 00:00:00


## 2. Define the temporal split

The competition asks you to predict purchases for the week **after** the training data ends.  
We simulate this by holding out the last 7 days as our validation set.

```
Training:    everything before SPLIT_DATE
Validation:  SPLIT_DATE  →  end of data
```

The last date in the data is 2020-09-22. So we split at 2020-09-16, giving us  
a 7-day validation window (9/16 – 9/22).

In [8]:
SPLIT_DATE = pd.Timestamp("2020-09-16")

train = transactions[transactions["t_dat"] < SPLIT_DATE].copy()
val   = transactions[transactions["t_dat"] >= SPLIT_DATE].copy()

print(f"Train: {len(train):,} transactions  ({train['t_dat'].min()} → {train['t_dat'].max()})")
print(f"Val:   {len(val):,} transactions  ({val['t_dat'].min()} → {val['t_dat'].max()})")

Train: 31,548,013 transactions  (2018-09-20 00:00:00 → 2020-09-15 00:00:00)
Val:   240,311 transactions  (2020-09-16 00:00:00 → 2020-09-22 00:00:00)


## 3. Build the validation ground truth

For each customer who made at least one purchase in the validation window,  
collect the set of article_ids they bought. This is what we score against.

In [9]:
val_ground_truth = (
    val
    .groupby("customer_id")["article_id"]
    .apply(set)
    .to_dict()
)

print(f"Customers with purchases in val window: {len(val_ground_truth):,}")

# Distribution of how many items each customer bought
counts = [len(v) for v in val_ground_truth.values()]
print(f"Items per customer — mean: {np.mean(counts):.1f}, median: {np.median(counts):.0f}, max: {max(counts)}")

Customers with purchases in val window: 68,984
Items per customer — mean: 3.1, median: 2, max: 50


## 4. MAP@12 evaluation function

This is the competition metric. Average Precision at 12 for each customer,  
averaged across all customers who had purchases in the validation window.

**Use this exact function for every model you build.**

In [10]:
def average_precision_at_k(predicted: list, actual: set, k: int = 12) -> float:
    """
    Compute AP@k for a single customer.

    Args:
        predicted: ordered list of predicted article_ids (length <= k)
        actual:    set of article_ids the customer actually bought
        k:         cutoff (12 for this competition)
    """
    if not actual:
        return 0.0

    predicted = predicted[:k]
    hits = 0
    sum_precision = 0.0

    for i, pred in enumerate(predicted):
        if pred in actual:
            hits += 1
            sum_precision += hits / (i + 1)

    return sum_precision / min(len(actual), k)


def map_at_k(predictions: dict, ground_truth: dict, k: int = 12) -> float:
    """
    Compute MAP@k across all customers.

    Args:
        predictions:  dict of {customer_id: [list of predicted article_ids]}
        ground_truth: dict of {customer_id: {set of actual article_ids}}
        k:            cutoff

    Returns:
        MAP@k score (float between 0 and 1)
    """
    ap_scores = []

    for customer_id, actual in ground_truth.items():
        predicted = predictions.get(customer_id, [])
        ap_scores.append(average_precision_at_k(predicted, actual, k))

    return np.mean(ap_scores) if ap_scores else 0.0

## 5. Sanity-check the metric with a dummy prediction

In [11]:
# Perfect prediction for one customer (should give AP = 1.0)
test_customer = list(val_ground_truth.keys())[0]
test_actual   = val_ground_truth[test_customer]

perfect_pred = list(test_actual)[:12]
ap = average_precision_at_k(perfect_pred, test_actual)
print(f"Perfect prediction AP@12: {ap:.4f}  (should be 1.0)")

# Completely wrong prediction
wrong_pred = ["0000000"] * 12
ap = average_precision_at_k(wrong_pred, test_actual)
print(f"Wrong prediction AP@12:   {ap:.4f}  (should be 0.0)")

# Partial hit: correct item at position 3
partial_pred = ["0000000", "0000001", perfect_pred[0]] + ["0000002"] * 9
ap = average_precision_at_k(partial_pred, test_actual)
print(f"One hit at pos 3 AP@12:   {ap:.4f}")

Perfect prediction AP@12: 1.0000  (should be 1.0)
Wrong prediction AP@12:   0.0000  (should be 0.0)
One hit at pos 3 AP@12:   0.3333


## 6. Save everything for downstream notebooks

In [12]:
train.to_parquet(OUT_DIR / "train.parquet", index=False)
val.to_parquet(OUT_DIR / "val.parquet", index=False)

print(f"✓ train.parquet  ({len(train):,} rows)")
print(f"✓ val.parquet    ({len(val):,} rows)")

✓ train.parquet  (31,548,013 rows)
✓ val.parquet    (240,311 rows)


## 7. Quick global popularity baseline (free score)

While we're here, let's get our very first MAP@12 number on the board.  
Recommend the 12 most-purchased items from the last 2 weeks of training data to everyone.

In [13]:
# Last 2 weeks of training data
recent_cutoff = SPLIT_DATE - pd.Timedelta(days=14)
recent_train  = train[train["t_dat"] >= recent_cutoff]

top_12_global = (
    recent_train["article_id"]
    .value_counts()
    .head(12)
    .index
    .tolist()
)

print("Global top-12 items (last 2 weeks of train):")
for i, aid in enumerate(top_12_global, 1):
    print(f"  {i:2d}. {aid}")

Global top-12 items (last 2 weeks of train):
   1. 0751471001
   2. 0909370001
   3. 0915526001
   4. 0751471043
   5. 0448509014
   6. 0706016001
   7. 0918292001
   8. 0918522001
   9. 0915529003
  10. 0896152002
  11. 0863595006
  12. 0865799006


In [14]:
# Score: give every val customer the same top-12
predictions_global = {cid: top_12_global for cid in val_ground_truth}

score = map_at_k(predictions_global, val_ground_truth)
print(f"\n{'='*40}")
print(f"  Global Popularity Baseline")
print(f"  MAP@12 = {score:.5f}")
print(f"{'='*40}")
print(f"\nThis is your floor. Every model from here should beat this.")


  Global Popularity Baseline
  MAP@12 = 0.00710

This is your floor. Every model from here should beat this.


---

## Summary

You now have:
- `data/split/train.parquet` — all transactions before 2020-09-16
- `data/split/val.parquet` — transactions from 2020-09-16 to 2020-09-22
- `map_at_k()` function ready to score any model
- Your first baseline score to beat

**Next step:** Copy the `average_precision_at_k` and `map_at_k` functions into a `src/eval.py`  
so every notebook can import them with `from src.eval import map_at_k`.